# Aegis — automatic Colab node setup

This notebook fetches **https://github.com/Terrificdatabytes/Aegis** (`main`) automatically. No ZIP upload is needed.

**Master:** https://aegis-q8r3.onrender.com/

Run one copy per node (`node-1` through `node-5`). Every copy uses the same software but must use its matching **per-node credential**, derived from the master secret currently configured on Render. Do not put the root master secret, operator password, or node secret in source code.

Run cells in order. To refresh the repository or restart an agent, run **Stop** first, then rerun setup. Colab is temporary demo compute; respect runtime quotas and disconnect after the session.

In [ ]:
# Change NODE_NAME in each notebook copy. Repository and master are preconfigured.
from pathlib import Path

NODE_NAME = "node-1" # @param ["node-1", "node-2", "node-3", "node-4", "node-5"]
REPO_URL = "https://github.com/Terrificdatabytes/Aegis.git"
REPO_BRANCH = "main"
MASTER_URL = "https://aegis-q8r3.onrender.com"
PROJECT_DIR = Path("/content/Aegis")
AGENT_PORT = 6000

# Optional: set a full commit SHA to freeze a rehearsed version. Leave empty for main.
COMMIT_SHA = ""

if NODE_NAME not in {f"node-{i}" for i in range(1, 6)}:
    raise ValueError("NODE_NAME must be node-1 through node-5.")
print(f"Configured {NODE_NAME}. Master: {MASTER_URL}")

## 1. Fetch the project and install agent dependencies

First run clones `main`. Reruns update a clean checkout with a fast-forward only; they never delete local edits. If a full commit SHA is supplied, setup checks out that fetched commit for rehearsal repeatability. The code revision is printed for traceability. Python, git and pip failures stop setup immediately.

In [ ]:
import os
import re
import subprocess
import sys

if "agent_process" in globals() and agent_process.poll() is None:
    raise RuntimeError("An agent is running. Run the Stop cell before updating code.")

if PROJECT_DIR.exists():
    if not (PROJECT_DIR / ".git").is_dir():
        raise RuntimeError(f"{PROJECT_DIR} already exists but is not a git checkout. Move it aside yourself.")
    origin = subprocess.check_output(
        ["git", "remote", "get-url", "origin"], cwd=PROJECT_DIR, text=True
    ).strip()
    if origin.removesuffix(".git").rstrip("/") != REPO_URL.removesuffix(".git").rstrip("/"):
        raise RuntimeError("Existing checkout has a different origin; refusing to modify it.")
    dirty = subprocess.check_output(
        ["git", "status", "--porcelain"], cwd=PROJECT_DIR, text=True
    ).strip()
    if dirty:
        raise RuntimeError("Checkout has local changes. Save or resolve them before updating; nothing was overwritten.")
else:
    PROJECT_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(PROJECT_DIR)], check=True)

subprocess.run(["git", "fetch", "origin", REPO_BRANCH], cwd=PROJECT_DIR, check=True)
if COMMIT_SHA:
    if not re.fullmatch(r"[0-9a-fA-F]{40}", COMMIT_SHA):
        raise ValueError("COMMIT_SHA must be a full 40-character commit SHA, or empty.")
    subprocess.run(["git", "cat-file", "-e", COMMIT_SHA + "^{commit}"], cwd=PROJECT_DIR, check=True)
    subprocess.run(["git", "checkout", "--detach", COMMIT_SHA], cwd=PROJECT_DIR, check=True)
else:
    subprocess.run(["git", "checkout", REPO_BRANCH], cwd=PROJECT_DIR, check=True)
    subprocess.run(["git", "merge", "--ff-only", "origin/" + REPO_BRANCH], cwd=PROJECT_DIR, check=True)

required = ["node-agent/agent.py", "node-agent/requirements.txt", "common/security.py", "scenarios"]
for relative in required:
    if not (PROJECT_DIR / relative).exists():
        raise FileNotFoundError(f"Expected project file missing: {relative}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(PROJECT_DIR / "node-agent/requirements.txt")], check=True)
revision = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=PROJECT_DIR, text=True).strip()
print(f"Project ready: {PROJECT_DIR}")
print(f"Code revision: {revision}")

## 2. Install Cloudflare Tunnel

Downloads the official Linux binary for the current CPU architecture. No Cloudflare account or tunnel token is needed for a temporary quick tunnel. Reruns reuse an installed binary. `latest` is convenient; specify an official release tag below if you need to pin it.

In [ ]:
import platform
import shutil
import urllib.request

CLOUDFLARED_RELEASE = "latest"  # Or an official release tag, such as a date-formatted version.
architecture = {"x86_64": "amd64", "aarch64": "arm64", "arm64": "arm64"}.get(platform.machine().lower())
if platform.system() != "Linux" or not architecture:
    raise RuntimeError("Use a supported Linux Colab runtime (x86-64 or ARM64).")

binary = shutil.which("cloudflared")
if not binary:
    release_path = "latest/download" if CLOUDFLARED_RELEASE == "latest" else f"download/{CLOUDFLARED_RELEASE}"
    download_url = f"https://github.com/cloudflare/cloudflared/releases/{release_path}/cloudflared-linux-{architecture}"
    destination = Path("/usr/local/bin/cloudflared")
    temporary = destination.with_suffix(".download")
    with urllib.request.urlopen(download_url, timeout=60) as response, temporary.open("wb") as output:
        shutil.copyfileobj(response, output)
    temporary.chmod(0o755)
    temporary.replace(destination)
    binary = str(destination)

subprocess.run([binary, "--version"], check=True)
print("Cloudflare Tunnel is ready. An installed binary is reused; replace it explicitly to change versions.")

## 3. Check master, load the node secret, and start

No master URL prompt is needed. For optional automatic secret loading, use Colab's **Secrets** panel (key icon), enable notebook access, and add the matching key:

| Node | Colab secret name |
|---|---|
| node-1 | `AEGIS_NODE_1_SECRET` |
| node-2 | `AEGIS_NODE_2_SECRET` |
| node-3 | `AEGIS_NODE_3_SECRET` |
| node-4 | `AEGIS_NODE_4_SECRET` |
| node-5 | `AEGIS_NODE_5_SECRET` |

Without a Colab secret, the notebook asks once using hidden password entry. The key is never printed. Only the derived node credential is passed to the child process; root/admin secrets are removed from its environment.

The master health check allows a short free-service cold start. It does not wake the service continuously or guarantee the reverse tunnel will work. The agent handles its own tunnel, registration, heartbeat and resource reporting.

In [ ]:
import getpass
import socket
import time
import requests

if "agent_process" in globals() and agent_process.poll() is None:
    raise RuntimeError("This notebook already has a running agent. Stop it before starting again.")

with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as probe:
    if probe.connect_ex(("127.0.0.1", AGENT_PORT)) == 0:
        raise RuntimeError(f"Port {AGENT_PORT} is occupied. Stop the existing agent or restart the runtime.")

print("Checking the configured Render master; a cold start may take a moment...")
deadline = time.monotonic() + 90
healthy = False
while time.monotonic() < deadline:
    try:
        response = requests.get(MASTER_URL + "/health", timeout=(5, 15), allow_redirects=False)
        if response.status_code == 200 and response.json().get("status") == "ok":
            healthy = True
            break
    except (requests.RequestException, ValueError):
        pass
    time.sleep(3)
if not healthy:
    raise RuntimeError("Master health check did not pass. Check Render, then rerun this cell; no agent was started.")

secret_name = "AEGIS_" + NODE_NAME.replace("-", "_").upper() + "_SECRET"
try:
    from google.colab import userdata
    node_secret = userdata.get(secret_name)
except Exception:
    node_secret = None
if not node_secret:
    node_secret = getpass.getpass(f"Matching per-node credential for {NODE_NAME} (hidden): ")
node_secret = node_secret.strip()
if len(node_secret) < 32:
    raise ValueError("Node credential is too short. Use the key derived for this node, not the master or operator password.")

env = {**os.environ, "NODE_NAME": NODE_NAME, "MASTER_URL": MASTER_URL,
       "NODE_SECRET": node_secret, "AGENT_PORT": str(AGENT_PORT), "PYTHONUNBUFFERED": "1"}
for variable in ["LOCAL_DEMO", "PUBLIC_URL", "MASTER_SECRET", "ADMIN_PASSWORD", "SESSION_SECRET"]:
    env.pop(variable, None)
LOG_PATH = Path(f"/content/{NODE_NAME}-agent.log")
log = LOG_PATH.open("w")
try:
    agent_process = subprocess.Popen(
        [sys.executable, str(PROJECT_DIR / "node-agent/agent.py")], cwd=PROJECT_DIR,
        env=env, stdout=log, stderr=subprocess.STDOUT, start_new_session=True
    )
finally:
    log.close()  # Child keeps its own stdout descriptor.
    env.pop("NODE_SECRET", None)
    del node_secret
print(f"Started {NODE_NAME}, PID {agent_process.pid}. Logs: {LOG_PATH}")
print("Run the next cell to check startup. Confirm the online node in the master dashboard.")

## 4. Check the agent and public tunnel

Checks local agent health and then the newly generated public tunnel. Shows diagnostic logs only after startup failure, or the last few lines for inspection. The dashboard is the source of truth for successful registration. If it stays offline, check the matching node credential and Render root secret. Wait for real resource samples; graphs do not backfill.

In [ ]:
if "agent_process" not in globals():
    raise RuntimeError("Run the start cell first.")

deadline = time.monotonic() + 75
public_url = None
while time.monotonic() < deadline:
    if agent_process.poll() is not None:
        print(LOG_PATH.read_text()[-4000:])
        raise RuntimeError(f"Agent exited with status {agent_process.returncode}. Inspect the logs above.")
    lines = LOG_PATH.read_text() if LOG_PATH.exists() else ""
    match = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", lines)
    if match:
        public_url = match.group(0)
        break
    time.sleep(2)

if not public_url:
    print(LOG_PATH.read_text()[-4000:])
    raise RuntimeError("No quick tunnel URL found. The agent may still be running. Inspect or stop it before retrying.")

for label, url in [("Local agent", f"http://127.0.0.1:{AGENT_PORT}"), ("Public tunnel", public_url)]:
    try:
        response = requests.get(url + "/health", timeout=(5, 15), allow_redirects=False)
        response.raise_for_status()
        health = response.json()
        if response.status_code != 200 or health.get("status") != "ok" or health.get("node_id") != NODE_NAME:
            raise ValueError("Unexpected health response or node identity")
        print(f"{label}: healthy ({NODE_NAME})")
    except (requests.RequestException, ValueError):
        print(f"{label}: not verified yet; wait briefly, then rerun this check.")

print("Master dashboard:", MASTER_URL)
print("Public tunnel:", public_url)
print("Confirm the node is online, then run a scenario from the dashboard.")
print("Recent agent logs:\n" + LOG_PATH.read_text()[-2000:])

## 5. Stop

Stops the agent **and its cloudflared child process group**, then escalates only if the group does not exit. Safe to run again. After the demo, disconnect/delete the Colab runtime. An agent restart receives a new quick-tunnel URL and registers it automatically.

In [ ]:
import signal

if "agent_process" not in globals():
    print("No agent was started by this notebook.")
else:
    # A tunnel child can outlive the parent, so signal the group even if the agent exited.
    group_id = agent_process.pid
    try:
        os.killpg(group_id, signal.SIGTERM)
        try:
            agent_process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            pass
        # Also give cloudflared time to exit if the parent exited first.
        group_deadline = time.monotonic() + 5
        while time.monotonic() < group_deadline:
            try:
                os.killpg(group_id, 0)
            except ProcessLookupError:
                break
            time.sleep(0.2)
        try:
            os.killpg(group_id, 0)
        except ProcessLookupError:
            pass
        else:
            os.killpg(group_id, signal.SIGKILL)
        agent_process.wait(timeout=5)
    except ProcessLookupError:
        pass
    finally:
        del agent_process  # Prevent a later cleanup cell from signalling a reused PID.
    print("Agent/tunnel stop requested. Disconnect the runtime when finished.")